In [ ]:
!pip install -q transformers peft albumentations imagehash
!pip uninstall -y -q torchao
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT = '/content/drive/MyDrive/thesis_ckpts'
os.makedirs(CKPT, exist_ok=True)
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
from google.colab import files
print("kaggle.json upload:")
files.upload()
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset -p /content/data/busi --unzip
!kaggle datasets download -d yeasirar/breast-lesions-usg-images-and-masks -p /content/data/breast --unzip
!kaggle datasets download -d orvile/bus-uclm-breast-ultrasound-dataset -p /content/data/uclm --unzip

In [ ]:
!find /content/data/busi -maxdepth 2 -type d
!echo "--- breast ---"
!find /content/data/breast -maxdepth 3 -type d
!find /content/data/breast -name "*.png" | head -6
!echo "--- uclm ---"
!find /content/data/uclm -maxdepth 3 -type d
!find /content/data/uclm -name "*.png" | head -6

In [ ]:
# cell 5 - normalize
from pathlib import Path
import numpy as np, re
from PIL import Image

OUT = Path('/content/breast_data'); OUT.mkdir(exist_ok=True)

def save_pair(name, img_path, mask_arr, dest):
    (dest/'images').mkdir(parents=True, exist_ok=True)
    (dest/'masks').mkdir(parents=True, exist_ok=True)
    Image.open(img_path).convert('RGB').save(dest/'images'/name)
    Image.fromarray((mask_arr > 127).astype(np.uint8) * 255).save(dest/'masks'/name)

# BUSI (train) : x.png + x_mask*.png, merge multiple masks
src = Path('/content/data/busi/Dataset_BUSI_with_GT')
dest = OUT/'BUSI'; n_multi = 0
for cls in ['benign', 'malignant']:              # 'normal' SKIPPED (empty masks)
    for img in sorted((src/cls).glob('*.png')):
        if '_mask' in img.name: continue
        parts = sorted((src/cls).glob(f'{img.stem}_mask*.png'))
        if not parts: print("no mask:", img.name); continue
        m = np.zeros_like(np.array(Image.open(parts[0]).convert('L')))
        for p in parts: m = np.maximum(m, np.array(Image.open(p).convert('L')))
        if len(parts) > 1: n_multi += 1
        save_pair(f"{cls}_{re.sub(r'[^0-9]','',img.stem)}.png", img, m, dest)
print(f"BUSI      : {len(list((dest/'images').iterdir()))} | multiple masks: {n_multi}")

# BrEaST (OOD) : caseNNN.png + caseNNN_tumor.png
src = Path('/content/data/breast/BrEaST-Lesions_USG-images_and_masks')
dest = OUT/'BrEaST'; skipped = 0
for img in sorted(src.glob('case*.png')):
    if '_tumor' in img.name or '_other' in img.name: continue
    tum = src/f'{img.stem}_tumor.png'
    if not tum.exists(): skipped += 1; continue     # normal tissue, no mask
    save_pair(f'{img.stem}.png', img, np.array(Image.open(tum).convert('L')), dest)
print(f"BrEaST    : {len(list((dest/'images').iterdir()))} | skipped, no mask: {skipped}")

# BUS-UCLM (OOD) : RGB mask (green = benign, red = malignant)
src = Path('/content/data/uclm/BUS-UCLM Breast ultrasound lesion segmentation dataset/'
           'BUS-UCLM Breast ultrasound lesion segmentation dataset/BUS-UCLM')
dest = OUT/'BUS-UCLM'; empty = 0
for mp in sorted((src/'masks').glob('*.png')):
    ip = src/'images'/mp.name
    if not ip.exists(): continue
    a = np.array(Image.open(mp).convert('RGB'))
    m = (a.max(axis=2) > 20).astype(np.uint8) * 255   # non-black = lesion
    if m.sum() == 0: empty += 1; continue             # normal tissue
    save_pair(mp.name, ip, m, dest)
print(f"BUS-UCLM  : {len(list((dest/'images').iterdir()))} | skipped, empty mask: {empty}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 4, figsize=(13, 9))
for r, name in enumerate(['BUSI', 'BrEaST', 'BUS-UCLM']):
    imgs = sorted((OUT/name/'images').iterdir())[:2]
    for c, p in enumerate(imgs):
        m = Image.open(OUT/name/'masks'/p.name)
        ax[r,c*2].imshow(Image.open(p)); ax[r,c*2].set_title(f'{name} {p.name}', fontsize=8)
        ax[r,c*2+1].imshow(m, cmap='gray'); ax[r,c*2+1].set_title('mask', fontsize=8)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#dedupe
import imagehash, itertools
from collections import defaultdict

def phashes(d):
    return {p.name: imagehash.phash(Image.open(p)) for p in sorted((d/'images').iterdir())}

H = {n: phashes(OUT/n) for n in ['BUSI', 'BrEaST', 'BUS-UCLM']}

# --- 1) internal duplicates within BUSI ---
THR = 5
ph = H['BUSI']
pairs = [(a, b) for a, b in itertools.combinations(ph, 2) if ph[a] - ph[b] <= THR]
par = {n: n for n in ph}
def find(x):
    while par[x] != x: par[x] = par[par[x]]; x = par[x]
    return x
for a, b in pairs: par[find(a)] = find(b)
grp = defaultdict(list)
for n in ph: grp[find(n)].append(n)
multi = [sorted(g) for g in grp.values() if len(g) > 1]
dups = {n for g in multi for n in g[1:]}      # keep first, drop the rest
print(f"BUSI internal duplicates: {len(multi)} groups, "
      f"{len(dups)} images dropped -> {len(ph)-len(dups)} remain")
for g in multi[:8]: print("   ", g)

# --- 2) train-test leakage: BUSI vs held-out OOD sets ---
keep = [n for n in ph if n not in dups]
for t in ['BrEaST', 'BUS-UCLM']:
    leak = [(a, b) for a in keep for b in H[t] if ph[a] - H[t][b] <= THR]
    print(f"BUSI <-> {t}: {len(leak)} near-duplicate pairs")
    for a, b in leak[:5]: print("   ", a, "<->", b)

np.save(f'{CKPT}/busi_dups.npy', sorted(dups), allow_pickle=True)

In [ ]:
# BUS-UCLM: 4 pairs have mismatched image/mask dimensions (cropped image,
# uncropped mask) — alignment unknown, so excluded rather than resized.
bad = [p.name for p in sorted((OUT/'BUS-UCLM'/'images').iterdir())
       if Image.open(p).size != Image.open(OUT/'BUS-UCLM'/'masks'/p.name).size]
im = [p for p in sorted((OUT/'BUS-UCLM'/'images').iterdir()) if p.name not in bad]
clean_sets['BUS-UCLM'] = (im, [OUT/'BUS-UCLM'/'masks'/p.name for p in im])
print(f"BUS-UCLM: dropped {len(bad)} -> {len(im)}")

for k, (i, m) in clean_sets.items():
    n = sum(Image.open(a).size != Image.open(b).size for a, b in zip(i, m))
    print(f"  {k:10s} {len(i):4d} | mismatched {n}")

In [ ]:
import matplotlib.pyplot as plt
check = [('benign_1.png','benign_318.png'), ('benign_105.png','benign_156.png'),
         ('benign_127.png','benign_37.png'), ('benign_128.png','benign_30.png')]
fig, ax = plt.subplots(len(check), 4, figsize=(13, 3*len(check)))
for r, (a, b) in enumerate(check):
    for c, n in enumerate([a, b]):
        ax[r,c*2].imshow(Image.open(OUT/'BUSI'/'images'/n)); ax[r,c*2].set_title(n, fontsize=8)
        ax[r,c*2+1].imshow(Image.open(OUT/'BUSI'/'masks'/n), cmap='gray')
        ax[r,c*2+1].set_title('mask', fontsize=8)
for a_ in ax.ravel(): a_.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import random, hashlib

dups = set(np.load(f'{CKPT}/busi_dups.npy', allow_pickle=True).tolist())

busi = [p for p in sorted((OUT/'BUSI'/'images').iterdir()) if p.name not in dups]
tr_n = int(len(busi) * 0.9)
idx = list(range(len(busi))); random.Random(42).shuffle(idx)      # seeded, order-independent
tr_files = [busi[i] for i in idx[:tr_n]]
va_files = [busi[i] for i in idx[tr_n:]]

def pairs(files, root):
    return [f for f in files], [root/'masks'/f.name for f in files]

train_imgs, train_masks = pairs(tr_files, OUT/'BUSI')
val_imgs,   val_masks   = pairs(va_files, OUT/'BUSI')

clean_sets = {}
for name in ['BrEaST', 'BUS-UCLM']:
    im = sorted((OUT/name/'images').iterdir())
    clean_sets[name] = (im, [OUT/name/'masks'/p.name for p in im])
clean_sets['BUSI-val'] = (val_imgs, val_masks)                    # in-domain reference

print(f"train {len(train_imgs)} | val {len(val_imgs)}")
for k, (i, m) in clean_sets.items(): print(f"  {k:10s} {len(i)}")

# md5 leak check on the exact files used (complements phash)
h = lambda ps: {hashlib.md5(p.read_bytes()).hexdigest() for p in ps}
tr_h = h(train_imgs)
for k, (i, _) in clean_sets.items():
    print(f"md5 overlap train <-> {k:10s}: {len(tr_h & h(i))}")

Start

In [ ]:
# cell 2 equivalent — identical constants to the polyp notebook
import random, numpy as np, math, os
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoModel
from peft import LoraConfig, get_peft_model, PeftModel

IMG_SIZE, PATCH_SIZE = 224, 14
BATCH_SIZE, VAL_SPLIT, SEED = 16, 0.1, 42
DEVICE = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print("Device:", DEVICE)

In [ ]:
# cell 3 equivalent — same Dataset class, breast file lists
class SegDataset(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False):
        self.image_paths, self.mask_paths = image_paths, mask_paths
        aug = [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5)] if augment else []
        self.transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *aug,
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, i):
        image = np.array(Image.open(self.image_paths[i]).convert('RGB'))
        mask  = np.array(Image.open(self.mask_paths[i]).convert('L'))
        t = self.transform(image=image, mask=mask)
        return t['image'].float(), (t['mask'].float()/255.0 > 0.5).float().unsqueeze(0)

train_loader = DataLoader(SegDataset(train_imgs, train_masks, augment=True),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(SegDataset(val_imgs, val_masks, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('train', len(train_imgs), '| val', len(val_imgs))
for k,(i,m) in clean_sets.items(): print(f'  {k:10s} {len(i)}')
x, y = next(iter(train_loader)); print(x.shape, y.shape, y.unique())

In [ ]:
# cell 4 equivalent
class LinearDecoder(nn.Module):
    def __init__(self, embed_dim=384):
        super().__init__()
        self.grid = IMG_SIZE // PATCH_SIZE
        self.proj = nn.Conv2d(embed_dim, 1, 1)
    def forward(self, tokens):
        B, N, D = tokens.shape
        x = tokens.permute(0,2,1).reshape(B, D, self.grid, self.grid)
        x = self.proj(x)
        return F.interpolate(x, size=IMG_SIZE, mode='bilinear', align_corners=False)

class DINOv2SegFrozen(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('facebook/dinov2-small')
        for p in self.encoder.parameters(): p.requires_grad = False
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        return self.decoder(self.encoder(pixel_values=x).last_hidden_state[:,1:,:])

class DINOv2SegFull(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('facebook/dinov2-small')
        for p in self.encoder.parameters(): p.requires_grad = True
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        return self.decoder(self.encoder(pixel_values=x).last_hidden_state[:,1:,:])

class DINOv2SegLoRAx(nn.Module):
    def __init__(self, rank=8, use_dora=False, target_modules=('query','value')):
        super().__init__()
        enc = AutoModel.from_pretrained('facebook/dinov2-small')
        cfg = LoraConfig(r=rank, lora_alpha=rank*2, target_modules=list(target_modules),
                         lora_dropout=0.1, bias='none', use_dora=use_dora)
        self.encoder = get_peft_model(enc, cfg)
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        return self.decoder(self.encoder(pixel_values=x).last_hidden_state[:,1:,:])

def dice_bce(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    num = 2*(p*target).sum(dim=(1,2,3)) + 1e-6
    den = p.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + 1e-6
    return bce + (1 - (num/den).mean())

@torch.no_grad()
def metrics_per_image(m, loader):
    """per-image DSC and IoU vectors (NOT batch-weighted)."""
    m.eval(); D, I = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast('cuda'):
            p = (torch.sigmoid(m(x)) > 0.5).float()
        inter = (p*y).sum(dim=(1,2,3))
        psum, ysum = p.sum(dim=(1,2,3)), y.sum(dim=(1,2,3))
        D.append(((2*inter+1e-6)/(psum+ysum+1e-6)).cpu())
        I.append(((inter+1e-6)/(psum+ysum-inter+1e-6)).cpu())
    return torch.cat(D).numpy(), torch.cat(I).numpy()

@torch.no_grad()
def eval_vec(m, imgs, masks):
    ld = DataLoader(SegDataset(imgs, masks, augment=False),
                    batch_size=16, shuffle=False, num_workers=2)
    return metrics_per_image(m, ld)

print("model classes ready")

In [ ]:
# cell 5 equivalent — LLRD + scheduler + trainer, identical to polyp
def make_warmup_cosine(opt, warmup_ep, total_ep, min_factor=0.01):
    def fn(ep):
        if ep < warmup_ep: return (ep+1)/max(1,warmup_ep)
        prog = (ep-warmup_ep)/max(1,total_ep-warmup_ep)
        return min_factor + (1-min_factor)*0.5*(1+math.cos(math.pi*prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def build_llrd_params(model_full, base_lr_enc, lr_dec, decay=0.75, wd=1e-4):
    enc = model_full.encoder; layers = enc.encoder.layer; n = len(layers)
    groups = [{'params':[p for p in enc.embeddings.parameters() if p.requires_grad],
               'lr': base_lr_enc*(decay**n), 'weight_decay': wd}]
    for i, layer in enumerate(layers):
        groups.append({'params':[p for p in layer.parameters() if p.requires_grad],
                       'lr': base_lr_enc*(decay**(n-1-i)), 'weight_decay': wd})
    groups.append({'params':[p for p in model_full.decoder.parameters() if p.requires_grad],
                   'lr': lr_dec, 'weight_decay': wd})
    return groups

def train_model(m, opt, tag, total_ep=20, warmup_ep=2, patience=5):
    sch = make_warmup_cosine(opt, warmup_ep, total_ep)
    scaler = torch.amp.GradScaler('cuda')
    best, bad = -1, 0
    for ep in range(total_ep):
        m.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = dice_bce(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        vd, _ = metrics_per_image(m, val_loader)
        v = vd.mean(); flag = ''
        if v > best:
            best, bad = v, 0
            torch.save(m.state_dict(), f'{CKPT}/{tag}.pth'); flag = ' *'
        else:
            bad += 1
        print(f"  ep {ep+1:2d}/{total_ep}  val DSC {v:.4f}{flag}", flush=True)
        if bad >= patience:
            print(f"  early stop (ep {ep+1})"); break
    m.load_state_dict(torch.load(f'{CKPT}/{tag}.pth'))
    return best

def train_or_load(m, opt, tag, **kw):
    p = f'{CKPT}/{tag}.pth'
    if os.path.exists(p):
        m.load_state_dict(torch.load(p))
        vd, _ = metrics_per_image(m, val_loader)
        print(f"  SKIP {tag} — val DSC {vd.mean():.4f}")
        return vd.mean()
    return train_model(m, opt, tag, **kw)

print("trainer ready")

In [ ]:
# Three strategies, seed 42 — same protocol as skin and polyp
models = {}

set_seed(42)
print("="*50, "\nFROZEN\n", "="*50)
m = DINOv2SegFrozen().to(DEVICE)
opt = torch.optim.AdamW(m.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
train_or_load(m, opt, 'frozen_breast_s42', total_ep=20, warmup_ep=2, patience=5)
models['frozen'] = m

set_seed(42)
print("="*50, "\nLoRA r=32\n", "="*50)
m = DINOv2SegLoRAx(rank=32).to(DEVICE)
opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
train_or_load(m, opt, 'lora_r32_breast_s42', total_ep=20, warmup_ep=2, patience=5)
models['lora_r32'] = m

set_seed(42)
print("="*50, "\nFULL FT\n", "="*50)
m = DINOv2SegFull().to(DEVICE)
opt = torch.optim.AdamW(build_llrd_params(m, 5e-6, 1e-4, decay=0.75))
train_or_load(m, opt, 'full_breast_s42', total_ep=20, warmup_ep=3, patience=5)
models['full'] = m

print(f"\n{'':10s} " + " ".join(f"{d:>12s}" for d in clean_sets))
for k, mdl in models.items():
    row = {d: eval_vec(mdl, *clean_sets[d])[0].mean() for d in clean_sets}
    print(f"{k:10s} " + " ".join(f"{row[d]:12.4f}" for d in clean_sets))

In [ ]:
# Lesion size distribution — is BUS-UCLM simply an easier set?
for k, (im, mk) in clean_sets.items():
    fr = [ (np.array(Image.open(p).convert('L')) > 127).mean() for p in mk ]
    print(f"{k:10s} lesion fraction  mean {np.mean(fr):.4f} | median {np.median(fr):.4f}")

In [ ]:
# Rank sweep — same protocol, seed 42
sweep = {}
for r in [4, 8, 16]:                      # r=32 already trained above
    set_seed(42)
    print("="*50, f"\nLoRA r={r}\n", "="*50)
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    train_or_load(m, opt, f'lora_r{r}_breast_s42', total_ep=20, warmup_ep=2, patience=5)
    sweep[r] = m
sweep[32] = models['lora_r32']

print(f"\n{'rank':>5s} " + " ".join(f"{d:>12s}" for d in clean_sets))
rank_dsc = {}
for r in [4, 8, 16, 32]:
    row = {d: eval_vec(sweep[r], *clean_sets[d])[0].mean() for d in clean_sets}
    rank_dsc[r] = row
    print(f"{r:5d} " + " ".join(f"{row[d]:12.4f}" for d in clean_sets))

full_dsc = {d: eval_vec(models['full'], *clean_sets[d])[0].mean() for d in clean_sets}
print(f"\nD(LoRA - full FT)")
for r in [4, 8, 16, 32]:
    print(f"{r:5d} " + " ".join(f"{rank_dsc[r][d]-full_dsc[d]:+12.4f}" for d in clean_sets))

np.save(f'{CKPT}/rank_sweep_breast.npy',
        {'rank_dsc': rank_dsc, 'full_dsc': full_dsc}, allow_pickle=True)

In [ ]:
# 40-epoch budget check: do low ranks catch up with more steps?
TOTAL_EP, PATIENCE = 40, 8
sweep40 = {}
for r in [4, 8, 16, 32]:
    set_seed(42)
    print("="*50, f"\nLoRA r={r} | 40ep\n", "="*50)
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    train_or_load(m, opt, f'lora_r{r}_breast_s42_e40',
                  total_ep=TOTAL_EP, warmup_ep=2, patience=PATIENCE)
    sweep40[r] = m

set_seed(42)
print("="*50, "\nFULL FT | 40ep\n", "="*50)
m_full40 = DINOv2SegFull().to(DEVICE)
opt = torch.optim.AdamW(build_llrd_params(m_full40, 5e-6, 1e-4, decay=0.75))
train_or_load(m_full40, opt, 'full_breast_s42_e40',
              total_ep=TOTAL_EP, warmup_ep=3, patience=PATIENCE)

print(f"\n=== 40ep ===\n{'rank':>5s} " + " ".join(f"{d:>12s}" for d in clean_sets))
rank40 = {}
for r in [4, 8, 16, 32]:
    row = {d: eval_vec(sweep40[r], *clean_sets[d])[0].mean() for d in clean_sets}
    rank40[r] = row
    print(f"{r:5d} " + " ".join(f"{row[d]:12.4f}" for d in clean_sets))
full40 = {d: eval_vec(m_full40, *clean_sets[d])[0].mean() for d in clean_sets}
print(f"{'full':>5s} " + " ".join(f"{full40[d]:12.4f}" for d in clean_sets))

print(f"\n=== rank effect (r16 - r4), 20ep vs 40ep ===")
for d in clean_sets:
    o = rank_dsc[16][d] - rank_dsc[4][d]
    n = rank40[16][d] - rank40[4][d]
    print(f"{d:10s} 20ep {o:+.4f} | 40ep {n:+.4f} | delta {n-o:+.4f}")

np.save(f'{CKPT}/rank_sweep_breast_e40.npy',
        {'rank_dsc': rank40, 'full_dsc': full40}, allow_pickle=True)

In [ ]:
import torch, numpy as np, os
from transformers import AutoModel
import matplotlib.pyplot as plt

@torch.no_grad()
def delta_w_spectra(ckpt_path, n_blocks=12):
    """Full FT weights minus pretrained weights; SVD of Q/K/V per block."""
    pre = AutoModel.from_pretrained('facebook/dinov2-small')
    pre_sd = {k: v for k, v in pre.state_dict().items()}
    sd = torch.load(ckpt_path, map_location='cpu')
    ft_sd = {k[len('encoder.'):]: v for k, v in sd.items() if k.startswith('encoder.')}
    out = {}
    for b in range(n_blocks):
        for name in ['query', 'key', 'value']:
            key = f'encoder.layer.{b}.attention.attention.{name}.weight'
            if key not in ft_sd or key not in pre_sd: continue
            dW = ft_sd[key].float() - pre_sd[key].float()
            out[(b, name)] = torch.linalg.svdvals(dW).numpy()
    del pre
    return out

def spectral_summary(s, thresholds=(0.90, 0.95, 0.99)):
    e = s**2
    c = np.cumsum(e) / e.sum()
    res = {f'r@{int(t*100)}': int(np.searchsorted(c, t) + 1) for t in thresholds}
    p = e / e.sum()
    res['eff_rank'] = float(np.exp(-(p * np.log(p + 1e-12)).sum()))
    res['s1_over_s10'] = float(s[0] / s[9]) if len(s) > 9 else np.nan
    return res

# Breast: 20-epoch checkpoint, to match how skin and polyp were measured
sp = delta_w_spectra(f'{CKPT}/full_breast_s42.pth')
rows = [spectral_summary(s) for s in sp.values()]
agg = {k: np.mean([r[k] for r in rows]) for k in rows[0]}
print(f"breast (20ep)  r@90 {agg['r@90']:.1f} | r@95 {agg['r@95']:.1f} | "
      f"r@99 {agg['r@99']:.1f} | eff_rank {agg['eff_rank']:.1f} | s1/s10 {agg['s1_over_s10']:.2f}")

# 40-epoch version too — checks whether the measure is budget-sensitive
if os.path.exists(f'{CKPT}/full_breast_s42_e40.pth'):
    sp40 = delta_w_spectra(f'{CKPT}/full_breast_s42_e40.pth')
    rows40 = [spectral_summary(s) for s in sp40.values()]
    agg40 = {k: np.mean([r[k] for r in rows40]) for k in rows40[0]}
    print(f"breast (40ep)  r@90 {agg40['r@90']:.1f} | r@95 {agg40['r@95']:.1f} | "
          f"r@99 {agg40['r@99']:.1f} | eff_rank {agg40['eff_rank']:.1f} | s1/s10 {agg40['s1_over_s10']:.2f}")

np.save(f'{CKPT}/spectra_breast.npy', {'e20': sp, 'agg20': agg}, allow_pickle=True)

**ResNet**

In [ ]:
# Stronger augmentation for from-scratch training (nnU-Net-style).
# Kept separate so the DINOv2 adaptation experiments stay unchanged.
class SegDatasetStrong(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False):
        self.image_paths, self.mask_paths = image_paths, mask_paths
        aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.25, rotate_limit=30,
                               border_mode=0, p=0.5),
            A.ElasticTransform(alpha=120, sigma=8, border_mode=0, p=0.2),
            A.RandomBrightnessContrast(0.3, 0.3, p=0.3),
            A.RandomGamma(gamma_limit=(70, 150), p=0.3),
            A.GaussNoise(p=0.2),
        ] if augment else []
        self.transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *aug,
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, i):
        image = np.array(Image.open(self.image_paths[i]).convert('RGB'))
        mask  = np.array(Image.open(self.mask_paths[i]).convert('L'))
        t = self.transform(image=image, mask=mask)
        return t['image'].float(), (t['mask'].float()/255.0 > 0.5).float().unsqueeze(0)

train_loader_s = DataLoader(SegDatasetStrong(train_imgs, train_masks, augment=True),
                            batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader_s   = DataLoader(SegDatasetStrong(val_imgs, val_masks, augment=False),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(train_loader_s.dataset)} | val {len(val_loader_s.dataset)}")

In [ ]:
# Residual-encoder U-Net trained from scratch, with deep supervision.
# nnU-Net Revisited (Isensee et al., MICCAI 2024) reports residual-encoder
# U-Nets as the strongest recipe for medical segmentation.
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.n1 = nn.InstanceNorm2d(cout, affine=True)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.n2 = nn.InstanceNorm2d(cout, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)
        self.skip = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                   nn.InstanceNorm2d(cout, affine=True))
                     if (stride != 1 or cin != cout) else nn.Identity())
    def forward(self, x):
        r = self.skip(x)
        x = self.act(self.n1(self.c1(x)))
        x = self.n2(self.c2(x))
        return self.act(x + r)

class ResUNet(nn.Module):
    def __init__(self, base=32, depth=5, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        ch = [base * 2**i for i in range(depth)]
        self.stem = ResBlock(3, ch[0])
        self.enc  = nn.ModuleList([ResBlock(ch[i], ch[i+1], stride=2) for i in range(depth-1)])
        self.up   = nn.ModuleList([nn.ConvTranspose2d(ch[i+1], ch[i], 2, 2) for i in reversed(range(depth-1))])
        self.dec  = nn.ModuleList([ResBlock(ch[i]*2, ch[i]) for i in reversed(range(depth-1))])
        self.head = nn.ModuleList([nn.Conv2d(ch[i], 1, 1) for i in reversed(range(depth-1))])
    def forward(self, x):
        feats = [self.stem(x)]
        for e in self.enc: feats.append(e(feats[-1]))
        d, outs = feats[-1], []
        for i, (u, dec, hd) in enumerate(zip(self.up, self.dec, self.head)):
            d = dec(torch.cat([u(d), feats[-2-i]], dim=1))
            outs.append(hd(d))
        full = F.interpolate(outs[-1], size=IMG_SIZE, mode='bilinear', align_corners=False)
        if not (self.training and self.deep_sup): return full
        return [full] + [F.interpolate(o, size=IMG_SIZE, mode='bilinear', align_corners=False)
                         for o in outs[:-1]]

def dice_bce_ds(out, target):
    """Deep supervision: weights halve per level, normalised."""
    if not isinstance(out, list): return dice_bce(out, target)
    w = [1 / 2**i for i in range(len(out))]
    w = [x / sum(w) for x in w]
    return sum(wi * dice_bce(o, target) for wi, o in zip(w, out))

_m = ResUNet(); print(f"ResUNet params: {sum(p.numel() for p in _m.parameters()):,}"); del _m

In [ ]:
def make_poly(opt, total_ep, power=0.9):
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: (1 - ep/total_ep)**power)

def train_scratch(m, opt, tag, total_ep=300, patience=50):
    sch = make_poly(opt, total_ep)
    scaler = torch.amp.GradScaler('cuda')
    best, bad = -1, 0
    for ep in range(total_ep):
        m.train()
        for x, y in train_loader_s:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = dice_bce_ds(m(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 12)
            scaler.step(opt); scaler.update()
        sch.step()
        vd, _ = metrics_per_image(m, val_loader_s)
        v = vd.mean(); flag = ''
        if v > best:
            best, bad = v, 0
            torch.save(m.state_dict(), f'{CKPT}/{tag}.pth'); flag = ' *'
        else:
            bad += 1
        if (ep+1) % 10 == 0 or flag:
            print(f"  ep {ep+1:3d}/{total_ep}  val DSC {v:.4f}{flag}", flush=True)
        if bad >= patience:
            print(f"  early stop (ep {ep+1})"); break
    m.load_state_dict(torch.load(f'{CKPT}/{tag}.pth'))
    return best

def train_or_load_scratch(m, opt, tag, **kw):
    p = f'{CKPT}/{tag}.pth'
    if os.path.exists(p):
        m.load_state_dict(torch.load(p))
        vd, _ = metrics_per_image(m, val_loader_s)
        print(f"  SKIP {tag} — val DSC {vd.mean():.4f}")
        return vd.mean()
    return train_scratch(m, opt, tag, **kw)

print("scratch trainer ready")

In [ ]:
#!rm -f /content/drive/MyDrive/thesis_ckpts/resunet_breast_lr0.003_e300_s42.pth

In [ ]:
# Same recipe as polyp: SGD lr=3e-3, momentum 0.99, poly LR, 300 epochs
set_seed(42)
m_res = ResUNet().to(DEVICE)
opt = torch.optim.SGD(m_res.parameters(), lr=3e-3, momentum=0.99,
                      nesterov=True, weight_decay=3e-5)
train_or_load_scratch(m_res, opt, 'resunet_breast_lr0.003_e300_s42',
                      total_ep=300, patience=50)

print(f"\n{'':10s} " + " ".join(f"{d:>12s}" for d in clean_sets))
row = {d: eval_vec(m_res, *clean_sets[d])[0].mean() for d in clean_sets}
print(f"{'ResUNet':10s} " + " ".join(f"{row[d]:12.4f}" for d in clean_sets))

In [ ]:
# Data-size sweep on breast: shrink the TRAINING set only; val and OOD sets stay fixed.
# Replicates the polyp subsampling experiment in a second domain, so the data-scale
# effect is not established from a single modality.
import random

# Capture the full training list once and never overwrite it
if 'FULL_TR_IMGS' not in globals():
    FULL_TR_IMGS, FULL_TR_MASKS = list(train_imgs), list(train_masks)
assert len(FULL_TR_IMGS) == 479, f"expected 479, got {len(FULL_TR_IMGS)}"

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_subset(frac, seed=42):
    """Identical subset for both models, so the comparison is paired."""
    idx = list(range(len(FULL_TR_IMGS)))
    random.Random(seed).shuffle(idx)
    idx = sorted(idx[:int(len(FULL_TR_IMGS) * frac)])
    return [FULL_TR_IMGS[i] for i in idx], [FULL_TR_MASKS[i] for i in idx]

# Epoch budgets scale inversely with subset size so every run gets comparable
# optimizer steps; early stopping terminates runs that converge sooner.
FRACS   = [0.25, 0.50]                    # 1.00 already trained for both models
EP_LORA = {25: 160, 50: 80,  100: 40}
EP_RES  = {25: 1200, 50: 600, 100: 300}
order   = ['BUSI-val', 'BrEaST', 'BUS-UCLM']

results_b = {}

for f in FRACS:
    im, mk = make_subset(f)
    pct = int(f * 100)
    print("="*60, f"\n{pct}% of training set ({len(im)} images) | "
          f"LoRA {EP_LORA[pct]}ep, ResUNet {EP_RES[pct]}ep\n", "="*60)

    # --- DINOv2 + LoRA r=32 ---
    train_loader = DataLoader(SegDataset(im, mk, augment=True), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    set_seed(42)
    m = DINOv2SegLoRAx(rank=32).to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    train_or_load(m, opt, f'lora_r32_breast_f{pct}_s42',
                  total_ep=EP_LORA[pct], warmup_ep=2, patience=8)
    results_b[(pct, 'lora')] = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    del m; torch.cuda.empty_cache()

    # ResUNet from scratch, aggressive augmentation (best low-data config on polyp)
    train_loader_s = DataLoader(SegDatasetStrong(im, mk, augment=True), batch_size=BATCH_SIZE,
                                shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    set_seed(42)
    m = ResUNet().to(DEVICE)
    opt = torch.optim.SGD(m.parameters(), lr=3e-3, momentum=0.99,
                          nesterov=True, weight_decay=3e-5)
    train_or_load_scratch(m, opt, f'resunet_breast_f{pct}_s42',
                          total_ep=EP_RES[pct], patience=50)
    results_b[(pct, 'res')] = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    del m; torch.cuda.empty_cache()

np.save(f'{CKPT}/datasize_breast.npy', results_b, allow_pickle=True)
print("\nsaved: datasize_breast.npy")

In [ ]:
OOD = ['BrEaST', 'BUS-UCLM']

# 100% figures from the runs already completed
results_b[(100,'res')]  = {'BUSI-val':0.824, 'BrEaST':0.646, 'BUS-UCLM':0.598}
results_b[(100,'lora')] = {'BUSI-val':0.803, 'BrEaST':0.713, 'BUS-UCLM':0.705}

print(f"{'%':>5s} {'n':>5s} | {'ResU in':>8s} {'DINO in':>8s} {'diff':>8s} "
      f"| {'ResU OOD':>9s} {'DINO OOD':>9s} {'diff':>8s}")
print('-'*72)
for pct in [25, 50, 100]:
    if (pct,'res') not in results_b: continue
    r, l = results_b[(pct,'res')], results_b[(pct,'lora')]
    ro, lo = np.mean([r[d] for d in OOD]), np.mean([l[d] for d in OOD])
    print(f"{pct:5d} {int(479*pct/100):5d} | {r['BUSI-val']:8.4f} {l['BUSI-val']:8.4f} "
          f"{r['BUSI-val']-l['BUSI-val']:+8.4f} | {ro:9.4f} {lo:9.4f} {ro-lo:+8.4f}")

print("\npolyp OOD gap for reference:  10% -0.191 | 25% -0.135 | 50% -0.132 | 100% -0.066")

BitFit

In [ ]:
for n in ['clean_sets','train_loader','val_loader','LinearDecoder','DINOv2SegFrozen',
          'DINOv2SegLoRAx','DINOv2SegFull','train_or_load','eval_vec','set_seed','CKPT']:
    print(f"{n:20s} {'OK' if n in globals() else 'MISSING'}")

In [ ]:
# cell 1
# BitFit (Ben Zaken et al., ACL 2022): all encoder weights frozen, biases trained.
class DINOv2SegBitFit(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('facebook/dinov2-small')
        for n, p in self.encoder.named_parameters():
            p.requires_grad = ('bias' in n)
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        return self.decoder(self.encoder(pixel_values=x).last_hidden_state[:,1:,:])

_m = DINOv2SegBitFit()
print(f"BitFit trainable: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
del _m

In [ ]:
#cell 2
# LR searched per domain: the thesis argument is that optimal settings are
# domain-dependent, so the polyp value is not transferred.
# Selection is on in-domain validation DSC; OOD scores are logged, not used.
LRS = [1e-2, 3e-3, 1e-3, 3e-4]
bf_val, bf_ood = {}, {}
for lr in LRS:
    set_seed(42)
    print("="*50, f"\nBitFit | lr={lr:.0e}\n", "="*50)
    m = DINOv2SegBitFit().to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=lr, weight_decay=1e-4)
    bf_val[lr] = train_or_load(m, opt, f'bitfit_breast_lr{lr:.0e}_s42',
                               total_ep=40, warmup_ep=2, patience=8)
    bf_ood[lr] = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in clean_sets}
    del m; torch.cuda.empty_cache()

print(f"\n{'lr':>8s} {'val':>8s} " + " ".join(f"{d:>12s}" for d in clean_sets))
for lr in LRS:
    print(f"{lr:8.0e} {bf_val[lr]:8.4f} " + " ".join(f"{bf_ood[lr][d]:12.4f}" for d in clean_sets))

BEST_LR = max(bf_val, key=bf_val.get)
print(f"\nselected on val: {BEST_LR:.0e}  (val {bf_val[BEST_LR]:.4f})")

In [ ]:
#cell 3# Capacity axis at a single budget: frozen -> BitFit -> LoRA r=4..32 -> full FT.
# All seed 42, 40 epochs. Frozen is trained here (existing one is 20-epoch).
set_seed(42)
m_fr = DINOv2SegFrozen().to(DEVICE)
opt = torch.optim.AdamW(m_fr.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
train_or_load(m_fr, opt, 'frozen_breast_s42_e40', total_ep=40, warmup_ep=2, patience=8)

set_seed(42)
m_bf = DINOv2SegBitFit().to(DEVICE)
opt = torch.optim.AdamW([p for p in m_bf.parameters() if p.requires_grad],
                        lr=BEST_LR, weight_decay=1e-4)
train_or_load(m_bf, opt, f'bitfit_breast_lr{BEST_LR:.0e}_s42',
              total_ep=40, warmup_ep=2, patience=8)

def n_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

cap = {}
cap['frozen'] = (n_trainable(m_fr), {d: eval_vec(m_fr, *clean_sets[d])[0] for d in clean_sets})
cap['BitFit'] = (n_trainable(m_bf), {d: eval_vec(m_bf, *clean_sets[d])[0] for d in clean_sets})

for r in [4, 8, 16, 32]:
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    m.load_state_dict(torch.load(f'{CKPT}/lora_r{r}_breast_s42_e40.pth'))
    cap[f'LoRA r={r}'] = (n_trainable(m), {d: eval_vec(m, *clean_sets[d])[0] for d in clean_sets})
    del m; torch.cuda.empty_cache()

m_f = DINOv2SegFull().to(DEVICE)
m_f.load_state_dict(torch.load(f'{CKPT}/full_breast_s42_e40.pth'))
cap['full FT'] = (n_trainable(m_f), {d: eval_vec(m_f, *clean_sets[d])[0] for d in clean_sets})

print(f"\n{'method':>10s} {'trainable':>11s} " + " ".join(f"{d:>12s}" for d in clean_sets))
for name, (npar, row) in cap.items():
    print(f"{name:>10s} {npar:11,d} " + " ".join(f"{row[d].mean():12.4f}" for d in clean_sets))

np.save(f'{CKPT}/capacity_breast_e40.npy',
        {k: {'params': v[0], **{d: v[1][d] for d in clean_sets}} for k, v in cap.items()},
        allow_pickle=True)

UNet

In [ ]:
for n in ['clean_sets','eval_vec','train_loader_s','val_loader_s','train_or_load_scratch',
          'dice_bce_ds','set_seed','CKPT','IMG_SIZE']:
    print(f"{n:22s} {'OK' if n in globals() else 'MISSING'}")
print('train size:', len(train_loader_s.dataset))

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.n1 = nn.InstanceNorm2d(cout, affine=True)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.n2 = nn.InstanceNorm2d(cout, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        x = self.act(self.n1(self.c1(x)))
        return self.act(self.n2(self.c2(x)))

class PlainUNet(nn.Module):
    def __init__(self, base=32, depth=5, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        ch = [base * 2**i for i in range(depth)]
        self.stem = PlainBlock(3, ch[0])
        self.enc  = nn.ModuleList([PlainBlock(ch[i], ch[i+1], stride=2) for i in range(depth-1)])
        self.up   = nn.ModuleList([nn.ConvTranspose2d(ch[i+1], ch[i], 2, 2) for i in reversed(range(depth-1))])
        self.dec  = nn.ModuleList([PlainBlock(ch[i]*2, ch[i]) for i in reversed(range(depth-1))])
        self.head = nn.ModuleList([nn.Conv2d(ch[i], 1, 1) for i in reversed(range(depth-1))])
    def forward(self, x):
        feats = [self.stem(x)]
        for e in self.enc: feats.append(e(feats[-1]))
        d, outs = feats[-1], []
        for i, (u, dec, hd) in enumerate(zip(self.up, self.dec, self.head)):
            d = dec(torch.cat([u(d), feats[-2-i]], dim=1))
            outs.append(hd(d))
        full = F.interpolate(outs[-1], size=IMG_SIZE, mode='bilinear', align_corners=False)
        if not (self.training and self.deep_sup): return full
        return [full] + [F.interpolate(o, size=IMG_SIZE, mode='bilinear', align_corners=False)
                         for o in outs[:-1]]

# LR carried from the polyp search (1e-3), exactly as ResUNet carries 3e-3.
set_seed(42)
m_pl = PlainUNet().to(DEVICE)
opt = torch.optim.SGD(m_pl.parameters(), lr=1e-3, momentum=0.99,
                      nesterov=True, weight_decay=3e-5)
train_or_load_scratch(m_pl, opt, 'plainunet_breast_lr0.001_e300_s42',
                      total_ep=300, patience=50)

print(f"\n{'':10s} " + " ".join(f"{d:>12s}" for d in clean_sets))
row = {d: eval_vec(m_pl, *clean_sets[d])[0].mean() for d in clean_sets}
print(f"{'PlainUNet':10s} " + " ".join(f"{row[d]:12.4f}" for d in clean_sets))

In [ ]:
set_seed(42)
m_pl3 = PlainUNet().to(DEVICE)
opt = torch.optim.SGD(m_pl3.parameters(), lr=3e-3, momentum=0.99,
                      nesterov=True, weight_decay=3e-5)
train_or_load_scratch(m_pl3, opt, 'plainunet_breast_lr0.003_e300_s42',
                      total_ep=300, patience=50)

print(f"\n{'':12s} " + " ".join(f"{d:>12s}" for d in clean_sets))
row = {d: eval_vec(m_pl3, *clean_sets[d])[0].mean() for d in clean_sets}
print(f"{'Plain 3e-3':12s} " + " ".join(f"{row[d]:12.4f}" for d in clean_sets))

In [ ]:
print(list(clean_sets.keys()))
print(CKPT)

In [ ]:
d0 = list(clean_sets)[0]
print("clean_sets key:", d0)
print("type:", type(clean_sets[d0]), "| len:", len(clean_sets[d0]))
for i, x in enumerate(clean_sets[d0]):
    print(f"  [{i}] {type(x).__name__}", end=" ")
    try: print(f"len={len(x)} first={x[0] if isinstance(x, list) else '...'}")
    except: print()

In [ ]:
import numpy as np, cv2, os

DOMAIN   = 'breast'
CAP_FILE = f'{CKPT}/capacity_breast_e40.npy'
REF      = 'LoRA r=32'
OUT      = f'{CKPT}/figexamples'
os.makedirs(OUT, exist_ok=True)

cap = np.load(CAP_FILE, allow_pickle=True).item()

for d in clean_sets:
    imgs, msks = clean_sets[d][0], clean_sets[d][1]
    dsc = np.asarray(cap[REF][d])
    assert len(dsc) == len(imgs), f"{d}: {len(dsc)} scores vs {len(imgs)} images"

    # median-scoring case: not the best, not the worst
    idx = int(np.argsort(dsc)[len(dsc) // 2])

    im = cv2.cvtColor(cv2.imread(imgs[idx]), cv2.COLOR_BGR2RGB)
    mk = cv2.imread(msks[idx], cv2.IMREAD_GRAYSCALE)
    mk = (mk > 127).astype(np.uint8)

    # keep the long side at 512 to bound file size
    s = 512 / max(im.shape[:2])
    if s < 1:
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        mk = cv2.resize(mk, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)

    np.savez_compressed(f'{OUT}/{DOMAIN}__{d.replace(" ","_").replace("*","star")}.npz',
                        image=im, mask=mk, dsc=float(dsc[idx]), setname=d, domain=DOMAIN)
    print(f"{d:14s} median DSC {dsc[idx]:.3f}  (n={len(dsc)})  idx {idx}")

In [ ]:
import numpy as np, matplotlib.pyplot as plt, glob, os
from matplotlib.patches import Rectangle

OUT = '/content/drive/MyDrive/thesis_ckpts/figexamples'

# row order: (domain, [column labels in order])  — first column is the training source
LAYOUT = [
    ('polyp',  ['Kvasir', 'CVC-ColonDB', 'ETIS', 'CVC-T']),
    ('breast', ['BUSI-val', 'BrEaST', 'BUS-UCLM']),
    ('skin',   ['ISIC 2016*', 'ISIC 2017*', 'PH2']),
]

def load(dom, name):
    f = f'{OUT}/{dom}__{name.replace(" ","_").replace("*","star")}.npz'
    return np.load(f, allow_pickle=True) if os.path.exists(f) else None

ncol = max(len(r[1]) for r in LAYOUT)
fig, axes = plt.subplots(len(LAYOUT), ncol, figsize=(3.1 * ncol, 3.4 * len(LAYOUT)))

for r, (dom, names) in enumerate(LAYOUT):
    for c in range(ncol):
        ax = axes[r, c]; ax.axis('off')
        if c >= len(names):
            continue
        z = load(dom, names[c])
        if z is None:
            ax.text(.5, .5, f'missing\n{names[c]}', ha='center', va='center', fontsize=8)
            continue
        ax.imshow(z['image'])
        ax.contour(z['mask'], levels=[0.5], colors=['#ffd400'], linewidths=2.0)
        train = (c == 0)
        ax.set_title(f"{names[c]}{'  (train source)' if train else ''}\nDSC {float(z['dsc']):.3f}",
                     fontsize=9, fontweight='bold' if train else 'normal')
        if train:
            ax.add_patch(Rectangle((0, 0), z['image'].shape[1] - 1, z['image'].shape[0] - 1,
                                   fill=False, ec='#2b6cb0', lw=3))
    axes[r, 0].set_ylabel(dom, fontsize=11)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/thesis_ckpts/fig_examples.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np, cv2, os

DOMAIN = 'breast'         # 'polyp' | 'breast' | 'skin'
OUT    = f'{CKPT}/figexamples'
os.makedirs(OUT, exist_ok=True)

# Midpoint of each set's file ordering, so the frame is not chosen on appearance.
# Overridden only where that frame carries an acquisition artefact.
IDX_OVERRIDE = {
    'BUS-UCLM': 140,      # midpoint frame carries a vendor overlay and calipers
}

for d in clean_sets:
    imgs, msks = clean_sets[d][0], clean_sets[d][1]
    idx = IDX_OVERRIDE.get(d) or len(imgs) // 2

    im = cv2.cvtColor(cv2.imread(str(imgs[idx])), cv2.COLOR_BGR2RGB)
    mk = (cv2.imread(str(msks[idx]), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    s = 512 / max(im.shape[:2])
    if s < 1:
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        mk = cv2.resize(mk, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)

    np.savez_compressed(
        f'{OUT}/{DOMAIN}__{d.replace(" ","_").replace("*","star")}.npz',
        image=im, mask=mk, setname=d, domain=DOMAIN, index=idx, n=len(imgs))
    print(f"{d:20s} idx {idx}  n={len(imgs)}")

In [ ]:
import matplotlib.pyplot as plt, cv2
imgs = clean_sets['BUS-UCLM'][0]
fig, ax = plt.subplots(2, 5, figsize=(16, 7))
for a, i in zip(ax.ravel(), range(120, 170, 5)):
    a.imshow(cv2.cvtColor(cv2.imread(str(imgs[i])), cv2.COLOR_BGR2RGB))
    a.set_title(i); a.axis('off')
plt.tight_layout(); plt.show()